In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

In [ ]:
load_dotenv()
api_key=os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

### Vector Store Creation

In [ ]:
vector_store = client.vector_stores.create(name="Industrial_taxonomy_space")
taxonomy_vector_id=vector_store.id
print(taxonomy_vector_id)

vs_6935676ca90881918749bdfa320f9c35


### Uploading to vector store

In [ ]:
vector_store_id = taxonomy_vector_id

# Path to your single JSON file
file_path = "/content/rebuild emotion taxanomy article 1.txt"

print(f"Uploading {file_path}...")

# Upload the single file
with open(file_path, "rb") as f:
    file_response = client.vector_stores.files.upload_and_poll(
        vector_store_id=vector_store_id,
        file=f
    )

# Check status (optional)
print(f"Uploaded file: Status - {file_response.status}")

print("Upload completed successfully!")


Uploading /content/rebuild emotion taxanomy article 1.txt...
Uploaded file: Status - completed
Upload completed successfully!


In [ ]:
article_text = """HCA Healthcare is one of the leading hospital chains in the U.S., with a significant presence across the country, although its business is concentrated primarily in Texas and Florida.\n\nThe industry is competitive, but HCA consistently performs well nonetheless. The company has several competitive advantages, including its extensive network of facilities that span acute care hospitals, specialty care, urgent care, and more. This diversified mix of facilities enables it to retain patients within its ecosystem, granting it significant negotiating power with private insurers and government payers.\n\nAs a result, HCA Healthcare has built relationships with these third-party institutions. Further, HCA routinely invests in cutting-edge technology, another factor that allows it to attract patients. These dynamics have enabled the company to increase its market share over time. It had a 24% market share in 2012, and 10 years later, that had risen to 27%.\n\nHCA also records consistent revenue and profits. In the second quarter, the company's top line reached $18.6 billion, representing a 6.4% increase compared to the same period last year. The company's earnings per share climbed 23.5% year over year to $6.83.\n\nOver the next decade, HCA Healthcare should benefit from increased hospital spending. In the meantime, the company will continue to implement the same strategy that has made it so successful over the years. It's also worth noting that there are significant regulatory and financial barriers to entry in the industry.\n\nNewcomers then have to compete with established corporations to attract patients and third-party payers, but it's challenging to do so because those, like HCA Healthcare, that have been around for a while have already built solid relationships with major players in the healthcare sector. That's why it's hard to imagine HCA Healthcare completely losing its position. The stock should continue performing well over the long run."

"""

### Retrieval from Vector store

In [ ]:
import json

# Load your emotion taxonomy JSON
taxonomy_file = "/content/rebuild emotion taxanomy article 1.txt"
with open(taxonomy_file, "r") as f:
    taxonomy = json.load(f)

# Build a lookup: Industry -> Sub-category Name -> Emotions
industry_lookup = {}
for entry in taxonomy:
    industry_name = entry["Industry"]
    sub_lookup = {}
    for sub in entry.get("Sub-categories", []):
        sub_name = sub["Name"]
        emotions = sub.get("Emotions", [])
        sub_lookup[sub_name] = emotions
    industry_lookup[industry_name] = sub_lookup

# Your existing search
results = client.vector_stores.search(
    vector_store_id="vs_6914573daa5c81918616d8a8da7351d4",
    query=article_text,
)

# Loop through results and enrich with emotions
for item in results.data:
    try:
        content_json = json.loads(item.content[0].text)
        industry = content_json.get("Industry", "")
        sub_category = content_json.get("Sub-category", "")

        # Get emotions from taxonomy
        emotions = industry_lookup.get(industry, {}).get(sub_category, [])

        # Add emotions to the result
        content_json["Emotions"] = emotions

        print(f"Score: {item.score:.3f}")
        print(json.dumps(content_json, indent=2))
        print("...\n")

    except json.JSONDecodeError:
        print(f"Score: {item.score:.3f}")
        print(item.content[0].text[:300], "...\n")


Score: 0.871
{
  "Industry": "Healthcare & Biotech",
  "Sub-category": "Medical Devices",
  "Description": "Diagnostic and therapeutic devices for hospital and home use.",
  "Emotions": []
}
...

Score: 0.869
{
  "Industry": "Healthcare & Biotech",
  "Sub-category": "Digital Health",
  "Description": "Software-enabled care delivery, monitoring, and patient engagement.",
  "Emotions": [
    {
      "Name": "Optimism",
      "Intensity": "Mild = positive outlook on technology adoption and patient engagement / Strong = high confidence in growth and innovation impact",
      "Direction": "Forward-looking",
      "Confidence": "High / Low"
    },
    {
      "Name": "Concern",
      "Intensity": "Mild = cautious about tech implementation risks / Strong = significant worry about cybersecurity, adoption challenges, or ROI",
      "Direction": "Forward-looking / Reactive",
      "Confidence": "High / Low"
    },
    {
      "Name": "Surprise",
      "Intensity": "Mild = unexpected tech breakth

In [ ]:
results_sorted = sorted(results.data, key=lambda x: x.score, reverse=True)

### Formatting Vector Retrieval

In [ ]:
def format_results(results):
    """
    Takes vector store search results and returns a formatted string
    containing Industry, Sub-category, and Emotions, one per line,
    suitable to feed into the LLM.
    """
    formatted_lines = []

    for item in results:
        for part in item.content:
            try:
                data = json.loads(part.text)
                industry = data.get("Industry")
                sub_category = data.get("Sub-category")

                if industry and sub_category:
                    # Retrieve emotions from taxonomy lookup
                    emotions_list = industry_lookup.get(industry, {}).get(sub_category, [])

                    # Convert list of dicts to readable strings
                    if emotions_list:
                        emotions_str = ", ".join([e.get("Name", "") for e in emotions_list])
                    else:
                        emotions_str = "No emotions found"

                    # Append formatted line
                    formatted_lines.append(
                        f"- {industry} (Sub-category: {sub_category}) | Emotions: {emotions_str}"
                    )
            except json.JSONDecodeError:
                continue

    return "\n".join(formatted_lines)
    # Usage example
formatted_results = format_results(results_sorted)

print("Candidates to feed into LLM:\n", formatted_results)



Candidates to feed into LLM:
 - Healthcare & Biotech (Sub-category: Medical Devices) | Emotions: No emotions found
- Healthcare & Biotech (Sub-category: Digital Health) | Emotions: Optimism, Concern, Surprise, Criticism, Relief, Neutral, Encouragement, Curiosity
- Healthcare & Biotech (Sub-category: Health Insurance) | Emotions: Optimism, Concern, Surprise, Criticism, Relief, Neutral, Encouragement, Curiosity
- Healthcare & Biotech (Sub-category: Remote Patient Monitoring (RPM)) | Emotions: No emotions found
- Healthcare & Biotech (Sub-category: Telemedicine) | Emotions: No emotions found
- Healthcare & Biotech (Sub-category: Clinical Trials) | Emotions: No emotions found
- Healthcare & Biotech (Sub-category: Gene Therapy & Genomics) | Emotions: No emotions found
- Healthcare & Biotech (Sub-category: Biomanufacturing / Bio-Production) | Emotions: No emotions found
- Healthcare & Biotech (Sub-category: Pharmaceuticals) | Emotions: No emotions found
- Healthcare & Biotech (Sub-category: 

### System Prompt

In [ ]:
SYSTEM_PROMPT = f"""
You are an industry-aware Emotion Classification Model.

Your job:
Given a news article AND the taxonomy evidence (industry → sub-category → emotion definitions), classify the SINGLE most central emotion expressed toward the already-identified industry and sub-category.

You DO NOT identify the industry.
Your ONLY tasks are:
1. Select the correct emotion label (Greed, Fear, Optimism, Anger, Relief, Uncertainity) from the taxonomy.
2. Select intensity, direction, and confidence values exactly as defined in the taxonomy.
3. Produce the final output mapped to the correct industry + sub-category.
4. Provide a short reason.

Strict Rules & Behavior Requirements:

────────────────────────────────────────────
【1. Use ONLY Taxonomy Evidence】
────────────────────────────────────────────
- You MUST use the exact strings for:
  • Industry
  • Sub-category
  • Emotion Name
  • Intensity / Direction / Confidence
- You may not invent new labels, intensities, or sub-categories.
- If the taxonomy does not support the emotion → you must use the closest valid mapped emotion.

────────────────────────────────────────────
【2. What Counts as Central / Emphasized Emotion】
────────────────────────────────────────────
An article expresses an emotion when it includes:
- Reactions to company performance
- Investor sentiment
- Risk warnings or concerns
- Optimism, momentum, excitement
- Market moves tied to expectations
- Fear, uncertainty, or caution around future outcomes

An emotion is **central** if:
- It is strongly emphasized
- It materially affects the industry or sub-category
- It is repeated or implicitly reinforced by context

Do NOT classify emotion if:
- The mention is only factual
- The reference does not convey sentiment
- It is background information unrelated to sentiment

────────────────────────────────────────────
【3. Relevance Levels】
────────────────────────────────────────────
Choose exactly one:
- HIGH: Emotion is a major theme or heavily emphasized.
- MEDIUM: Emotion is present but secondary.
- LOW: Emotion is brief but meaningful.

────────────────────────────────────────────
【4. Reasoning Rules】
────────────────────────────────────────────
- Keep the reason very short and factual.
- If the emotion is supported by retrieved taxonomy evidence:
    End the reason with: **Retrieved from vector store**
- If there is NO supported emotion and you infer the closest valid one:
    End the reason with: **Inferred**
- Never mix both.

────────────────────────────────────────────
【5. Disambiguation Logic】
────────────────────────────────────────────
Map emotion wording to taxonomy categories:
- “optimism, growth, rally, gains, demand” → Positive
- “concerns, risk, warning, instability, decline” → Negative
- “mixed, muted, balanced, flat” → Neutral

If multiple emotions appear:
- Choose the most dominant one for the matched industry + sub-category only.
- Ignore unrelated industries.

────────────────────────────────────────────
【6. Hallucination Guardrails】
────────────────────────────────────────────
- Use ONLY article text + taxonomy evidence.
- Do NOT invent events, numbers, partnerships, technologies, or industries.
- Do NOT infer more than ONE emergent emotion.
- Do NOT use emotional terms that do not exist in the taxonomy.
- Do NOT overfit the article — classify only what is clearly supported.

When producing the final JSON output:

- ALWAYS return **all relevant emotions** found in the article, not just the strongest one.
- The field "shortlisted_emotions" must be an ARRAY and may contain:
    • 1 emotion (if only one clear emotion exists), OR
    • multiple emotions (if the article expresses several emotional tones).

Rules for multiple emotions:
- Each emotion must independently include: industry, sub_category, emotion_label, intensity, direction, confidence, relevance, reason.
- Emotions cannot contradict each other; choose distinct emotional signals.
- Do NOT merge emotions into a single output.
- Only output an emotion if it clearly appears in the article or in the retrieved taxonomy evidence.


────────────────────────────────────────────
【7. Output Requirements】
────────────────────────────────────────────
Output a JSON object in the exact structure:

{{
  "shortlisted_emotions": [
    {{
      "industry": "<Industry from taxonomy>",
      "sub_category": "<Sub-category from taxonomy>",
      "emotion_label": "<(Greed, Fear, Optimism, Anger, Relief, Uncertainity>",
      "intensity": "<Value from taxonomy>",
      "direction": "<Value from taxonomy>",
      "confidence": "<Value from taxonomy>",
      "relevance": "<HIGH/MEDIUM/LOW>",
      "reason": "<Short explanation ending with either 'Retrieved from vector store' or 'Inferred'>"
    }}
  ],
  "emergent_emotion_candidate": null
}}
"""


### User Prompt

In [ ]:
USER_PROMPT=f"""
Article:
{article_text}

Retrieved Taxonomy Evidence:
{formatted_results}

Task:
- Shortlist only CENTRAL / EMPHASIZED emotions relevant to the industry and sub-category found in the retrieved evidence.
- For each shortlisted item return:
    industry,
    sub_category,
    emotion_label (),
    intensity,
    direction,
    confidence,
    relevance (LOW/MEDIUM/HIGH),
    reason.

  - If the item is taken directly from the evidence, the reason MUST end with: "Retrieved from vector store".
  - If no emotion is relevant in evidence, you may infer ONE emergent emotion, returned under emergent_emotion_candidate, and the reason must end with: "Inferred".

- Answer deterministically. Identical input must always yield identical output.
-Use only the article text and the provided taxonomy; do not hallucinate events, numbers, sub-categories or companies.
-do not add anything that is not in taxanomy, only retrieve sub categories form taxanomy

"""

### JSON Schema

In [ ]:
textt_format = {
    "format": {
        "type": "json_schema",
        "name": "EmotionClassification",
        "schema": {
            "type": "object",
            "properties": {
                "shortlisted_emotions": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "industry": {"type": "string"},
                            "sub_category": {"type": "string"},
                            "emotion_label": {
                                "type": "string",
                                "enum": ["Fear", "Greed", "Optimism", "Uncertainty", "Relief", "Anger"]
                            },
                            "intensity": {"type": "string"},
                            "direction": {"type": "string"},
                            "confidence": {"type": "string"},
                            "relevance": {
                                "type": "string",
                                "enum": ["LOW", "MEDIUM", "HIGH"]
                            },
                            "reason": {"type": "string"}
                        },
                        "required": [
                            "industry",
                            "sub_category",
                            "emotion_label",
                            "intensity",
                            "direction",
                            "confidence",
                            "relevance",
                            "reason"
                        ],
                        "additionalProperties": False
                    }
                },
                "emergent_emotion_candidate": {
                    "anyOf": [
                        {
                            "type": "object",
                            "properties": {
                                "emotion_label": {
                                    "type": "string",
                                    "enum": ["Fear", "Greed", "Optimism", "Uncertainty", "Relief", "Anger"]
                                },
                                "intensity": {"type": "string"},
                                "direction": {"type": "string"},
                                "confidence": {"type": "string"},
                                "Description": {"type": "string"}
                            },
                            "required": ["emotion_label", "intensity", "direction", "confidence", "rationale"],
                            "additionalProperties": False
                        },
                        {"type": "null"}
                    ]
                }
            },
            "required": ["shortlisted_emotions", "emergent_emotion_candidate"],
            "additionalProperties": False
        },
        "strict": False
    }
}


### LLM Call

In [ ]:
response = client.responses.create(
        model="gpt-5-mini",
        input=[
            {"role": "system", "content": SYSTEM_PROMPT },
            {"role": "user", "content": USER_PROMPT}
        ],
        text=textt_format,

)
result=response.output_text
print(result)


{
  "shortlisted_emotions": [
    {
      "industry": "Healthcare & Biotech",
      "sub_category": "Health Insurance",
      "emotion_label": "Optimism",
      "intensity": "High",
      "direction": "Positive",
      "confidence": "High",
      "relevance": "HIGH",
      "reason": "Strong positive outlook from market-share gains, revenue and profit growth and continued long-term performance. Retrieved from vector store"
    }
  ],
  "emergent_emotion_candidate": null
}


In [ ]:
try:
    parsed = json.loads(result)

    pretty_json = json.dumps(parsed, indent=4, ensure_ascii=False)

    print(pretty_json)

except json.JSONDecodeError as e:
    print("Invalid JSON:", e)

{
    "shortlisted_emotions": [
        {
            "industry": "Healthcare & Biotech",
            "sub_category": "Health Insurance",
            "emotion_label": "Optimism",
            "intensity": "High",
            "direction": "Positive",
            "confidence": "High",
            "relevance": "HIGH",
            "reason": "Strong positive outlook from market-share gains, revenue and profit growth and continued long-term performance. Retrieved from vector store"
        }
    ],
    "emergent_emotion_candidate": null
}
